# 03 — Exploratory Data Analysis

Input: `philly311_clean.csv` from notebook 02.

Goal: understand the data well enough to make informed choices in modeling (the fast/slow
threshold, which features look promising, whether the log transform was warranted).
Every chart here should end with a one-line takeaway you can reuse in your report.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

TYPE_COL = "service_name"       # keep consistent with notebook 02's CONFIG
ZIP_COL = "zipcode"
AGENCY_COL = "agency_responsible"

df = pd.read_csv("philly311_clean.csv", parse_dates=["requested_datetime", "closed_datetime"])
df.describe(include="all").T

## 1. Distribution of resolution time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["resolution_hours"], bins=60, ax=axes[0])
axes[0].set_title("Resolution time (hours) — raw")
sns.histplot(df["log_resolution_hours"], bins=60, ax=axes[1])
axes[1].set_title("Resolution time — log1p transformed")
plt.tight_layout()
plt.show()

print(df["resolution_hours"].describe())
print("\nMedian (hours):", df["resolution_hours"].median())
print("Median (days):", df["resolution_hours"].median() / 24)

*Note the median above — you'll use it (or another cutoff you justify) for the
classification threshold in notebook 04.*

## 2. Resolution time by day of week / weekend

In [ ]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
plt.figure(figsize=(9, 4))
sns.boxplot(data=df, x="day_of_week", y="log_resolution_hours", order=order, showfliers=False)
plt.title("Log resolution time by day of week (requested)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 3. Resolution time by responsible agency

In [ ]:
top_agencies = df[AGENCY_COL].value_counts().head(10).index
plt.figure(figsize=(9, 5))
sns.boxplot(
    data=df[df[AGENCY_COL].isin(top_agencies)],
    y=AGENCY_COL, x="log_resolution_hours", showfliers=False
)
plt.title("Log resolution time by agency (top 10 by volume)")
plt.tight_layout()
plt.show()

## 4. Resolution time by location (zip code)

In [ ]:
zip_summary = (
    df.groupby(ZIP_COL)["resolution_hours"]
    .agg(["median", "count"])
    .query("count >= 30")
    .sort_values("median", ascending=False)
)
zip_summary.head(15)

## 5. Volume over time

In [ ]:
monthly = df.set_index("requested_datetime").resample("MS").size()
plt.figure(figsize=(10, 4))
monthly.plot()
plt.title("Requests per month")
plt.ylabel("count")
plt.tight_layout()
plt.show()

## 6. If your subset has sub-types, compare them

In [ ]:
if df[TYPE_COL].nunique() > 1:
    plt.figure(figsize=(9, 5))
    sns.boxplot(data=df, y=TYPE_COL, x="log_resolution_hours", showfliers=False)
    plt.title("Log resolution time by request sub-type")
    plt.tight_layout()
    plt.show()
else:
    print("Only one type selected — skip this comparison, or go back and widen CHOSEN_TYPES in notebook 01.")

## Write these down for your report
Fill in your own real observations here once you've run the cells above, e.g.:
- Which factor showed the clearest relationship with resolution time?
- Was the log transform actually justified by the raw-vs-log histograms?
- Any surprising pattern (e.g. a specific zip code or agency being a clear outlier)?
- Does volume have a seasonal pattern that might explain slow periods?

These become your EDA section and directly inform the feature choices in notebook 04.